In [8]:
%config IPCompleter.use_jedi = False

In [3]:
import io
import pathlib
from zipfile import ZipFile

import frontmatter
import requests

# Intro

In [2]:
test = frontmatter.loads("""---
title: "RAG-documentation-agent"
author: "John Doe"
---
# Introduction

This is a very borring intro.
""")

In [3]:
test.metadata

{'title': 'RAG-documentation-agent', 'author': 'John Doe'}

In [4]:
test.content

'# Introduction\n\nThis is a very borring intro.'

In [5]:
test.to_dict()

{'title': 'RAG-documentation-agent',
 'author': 'John Doe',
 'content': '# Introduction\n\nThis is a very borring intro.'}

# Hands on

In [5]:
OWNER="evidentlyai"
REPO="docs"
BRANCH="main"

In [72]:
OWNER="fastapi"
REPO="typer"
BRANCH="master"

In [6]:
def download_zip_file(owner: str, repo: str, branch: str = "main"):
    url = f"https://codeload.github.com/{owner}/{repo}/zip/refs/heads/{branch}"
    return requests.get(url=url)

In [7]:
response = download_zip_file(owner=OWNER, repo=REPO, branch=BRANCH)
response.url, response.status_code

('https://codeload.github.com/evidentlyai/docs/zip/refs/heads/main', 200)

In [9]:
def getFilesWithMetadata(content: str,*, extensions: list[str], folder: str | None = None):
    with ZipFile(io.BytesIO(content)) as zfile:
        for file_info in zfile.infolist():
            filename = file_info.filename.lower()
            if folder:
                parts = filename.strip("/").split("/")
                if folder not in parts:
                    continue
            if pathlib.Path(filename).suffix in set(extensions):
                with zfile.open(file_info, mode="r") as file:
                    md_content = file.read()
                    fm_content = frontmatter.loads(text=md_content)
                    fm_content_dict = fm_content.to_dict()
                    fm_content_dict["filename"] = filename
                    yield fm_content_dict

In [10]:
fm_contents = [fm_content for fm_content in getFilesWithMetadata(response.content, extensions=[".md"])]

In [11]:
fm_contents[1].keys()

dict_keys(['content', 'filename'])

In [12]:
def get_files_from_repo(owner: str, repo: str, extensions: list[str], branch: str = "main", folder: str | None = None):
    response = download_zip_file(owner=owner, repo=repo, branch=branch)
    yield from getFilesWithMetadata(response.content, extensions=extensions, folder=folder)

In [15]:
fm_contents = [fm_content for fm_content in get_files_from_repo(owner=OWNER, repo=REPO, extensions=[".md", ".mdx"], branch=BRANCH, folder=None)]

In [16]:
fm_contents[1], len(fm_contents)

({'title': 'Delete Plant',
  'openapi': 'DELETE /plants/{id}',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/delete.mdx'},
 95)